# Email Priority Classification System
## Exploratory Data Analysis & Model Training

This notebook walks through the full pipeline for classifying emails into four priority levels:
- **High Priority** — urgent issues requiring immediate action
- **Medium Priority** — regular work communication
- **Low Priority** — informational or optional content
- **Spam** — promotional, phishing, or unwanted messages

The pipeline uses TF-IDF feature extraction paired with Multinomial Naive Bayes and Logistic Regression classifiers.

In [ ]:
# ── Imports and path setup ──────────────────────────────────────────────────
import sys
import os
import warnings
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 110

# Add project root so we can import data and model packages
PROJECT_ROOT = r'd:\PROGRAMMING\ML_Projects\Email_Priotization_System_Using_NLP'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Python:', sys.version)
print('Pandas:', pd.__version__)
print('NumPy: ', np.__version__)

In [ ]:
# ── Load dataset ────────────────────────────────────────────────────────────
from data.preprocess import load_dataset

df = load_dataset()
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# ── Class distribution ──────────────────────────────────────────────────────
counts = df['label'].value_counts()
print('Class distribution:')
print(counts.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colours = ['#dc3545', '#fd7e14', '#0d6efd', '#6c757d']
bars = ax.bar(counts.index, counts.values, color=colours, edgecolor='white', linewidth=1.2)
ax.set_title('Email Label Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Priority Class', fontsize=11)
ax.set_ylabel('Number of Samples', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Text preprocessing demo ─────────────────────────────────────────────────
from data.preprocess import clean_text

sample_subject = 'URGENT: Production server is DOWN - immediate action required!'
sample_body    = 'The production server crashed 10 minutes ago. All customer services are affected. Please escalate.'
sample_raw     = sample_subject + ' ' + sample_body

print('=== Raw text ===')
print(sample_raw)
print()
print('=== After clean_text() ===')
print(clean_text(sample_raw))

In [ ]:
# ── TF-IDF feature extraction ───────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print('Preprocessing all texts...')
df['clean_text'] = df['text'].apply(clean_text)

le = LabelEncoder()
y  = le.fit_transform(df['label'])
print('Classes:', list(le.classes_))

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_text'], y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vectorizer.fit_transform(X_train_raw)
X_test  = vectorizer.transform(X_test_raw)

print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')

In [ ]:
# ── Train both classifiers ──────────────────────────────────────────────────
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Multinomial Naive Bayes
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train, y_train)
nb_preds = nb.predict(X_test)
print('=== Multinomial Naive Bayes ===')
print(f'Accuracy: {accuracy_score(y_test, nb_preds):.4f}')
print(classification_report(y_test, nb_preds, target_names=le.classes_))

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0, solver='lbfgs')
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
print('=== Logistic Regression ===')
print(f'Accuracy: {accuracy_score(y_test, lr_preds):.4f}')
print(classification_report(y_test, lr_preds, target_names=le.classes_))

In [ ]:
# ── Confusion matrix heatmap ────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

# Use Logistic Regression predictions for the heatmap
cm     = confusion_matrix(y_test, lr_preds)
labels = le.classes_

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)

thresh = cm.max() / 2.0
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# ── Sample predictions ───────────────────────────────────────────────────────
test_emails = [
    {'subject': 'URGENT: Database is down', 'body': 'All services are failing. Need immediate fix.'},
    {'subject': 'Sprint review this Friday', 'body': 'Please prepare your demo for the sprint review meeting.'},
    {'subject': 'FYI: Office closed Monday', 'body': 'The office will be closed on Monday for a public holiday.'},
    {'subject': 'Claim your free prize now!', 'body': 'You have been selected. Click to claim your $1000 reward.'},
    {'subject': 'Q3 roadmap feedback needed', 'body': 'Could you review the Q3 roadmap draft and share feedback by Thursday?'},
]

# Build feature matrix for the test emails
test_texts    = [clean_text(e['subject'] + ' ' + e['body']) for e in test_emails]
test_features = vectorizer.transform(test_texts)
test_preds    = lr.predict(test_features)
test_proba    = lr.predict_proba(test_features)

print(f"{'Subject':<45} {'Prediction':<18} {'Confidence':>10}")
print('-' * 76)
for email, pred_idx, proba in zip(test_emails, test_preds, test_proba):
    label = le.classes_[pred_idx]
    conf  = proba.max()
    print(f"{email['subject'][:44]:<45} {label:<18} {conf:>9.1%}")

## Conclusions

- Both Multinomial Naive Bayes and Logistic Regression perform well on the synthetic dataset.
- Logistic Regression typically edges ahead slightly in accuracy and is selected as the production model.
- TF-IDF with bigrams captures meaningful phrases (e.g. *urgent action*, *click here*) that are strong discriminators between classes.
- The preprocessing pipeline (HTML stripping, lemmatisation, stopword removal) ensures clean, consistent features.
- For production use, training on a larger real-world corpus (e.g. Enron) would further improve generalisation.

### Next steps
- Integrate a transformer-based model (e.g. DistilBERT) for higher accuracy.
- Add active learning to continuously improve the model with user feedback.
- Extend the label schema to include sub-priorities or custom user-defined categories.